In [1]:
#import statements
import os
import glob
import random
import numpy as np
import pandas as pd
import pydicom
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from torchvision import models, transforms
from tqdm import tqdm

ModuleNotFoundError: No module named 'pydicom'

In [ ]:
# =========================================================
# 1. CONFIG
# =========================================================
CSV_PATH = "cleaned_calc_description_train.csv"
DATA_ROOT = "../data"

VIEW_TO_USE = "CC"   # change to "MLO" later
BATCH_SIZE = 16
NUM_EPOCHS = 8
LR = 1e-4
IMG_SIZE = 224
NUM_WORKERS = 2
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# =========================================================
# 2. REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
# =========================================================
# 3. LOAD CSV
# =========================================================
df = pd.read_csv(CSV_PATH)

print("Columns:")
print(df.columns.tolist())

# Make column names easier to work with
df.columns = [c.strip() for c in df.columns]

# Some files say "assessment", some may be misspelled like "assesment"
target_col = None
for c in df.columns:
    if c.lower() == "assessment":
        target_col = c
        break
    if c.lower() == "assesment":
        target_col = c
        break

if target_col is None:
    raise ValueError("Could not find 'assessment' column in the CSV.")

# Normalize likely column names
patient_col = "patient_id"
view_col = "image view"
path_col = "image file path"

required_cols = [patient_col, view_col, path_col, target_col]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

In [ ]:
# =========================================================
# 4. FILTER TO ONE VIEW
# =========================================================
df[view_col] = df[view_col].astype(str).str.strip().str.upper()
df = df[df[view_col] == VIEW_TO_USE].copy()

print(f"\nRows after filtering to {VIEW_TO_USE}: {len(df)}")

In [ ]:
# =========================================================
# 5. CLEAN TARGET (BI-RADS / assessment)
# =========================================================
# Keep only rows with a valid target
df = df.dropna(subset=[target_col]).copy()

# Convert assessment to integer if possible
def clean_assessment(x):
    try:
        return int(float(x))
    except:
        return np.nan

df[target_col] = df[target_col].apply(clean_assessment)
df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(int)

print("\nAssessment value counts:")
print(df[target_col].value_counts().sort_index())

# Map BI-RADS labels to 0...K-1
classes = sorted(df[target_col].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

df["label"] = df[target_col].map(class_to_idx)

print("\nClass mapping:")
print(class_to_idx)

In [ ]:
# =========================================================
# 6. RESOLVE RAW IMAGE PATH
# =========================================================
# Your cleaned CSV now stores only UID2 in "image file path"
# We will find the .dcm file inside ../data/<UID2>/

def resolve_dicom_path(uid_folder, data_root=DATA_ROOT):
    if pd.isna(uid_folder):
        return None

    folder = os.path.join(data_root, str(uid_folder))
    if not os.path.isdir(folder):
        return None

    dcms = glob.glob(os.path.join(folder, "*.dcm"))
    if len(dcms) == 0:
        return None

    # usually there is just one raw image .dcm in that folder
    return dcms[0]

df["dicom_path"] = df[path_col].apply(resolve_dicom_path)

before = len(df)
df = df.dropna(subset=["dicom_path"]).copy()
after = len(df)

print(f"\nRows with resolvable DICOM path: {after} / {before}")

In [ ]:
# =========================================================
# 7. GROUPED TRAIN/VAL SPLIT BY PATIENT
# =========================================================
groups = df[patient_col].astype(str).values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(gss.split(df, groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print(f"\nTrain rows: {len(train_df)}")
print(f"Val rows:   {len(val_df)}")

train_patients = set(train_df[patient_col].astype(str))
val_patients = set(val_df[patient_col].astype(str))
overlap = train_patients.intersection(val_patients)

print(f"Patient overlap between train and val: {len(overlap)}")
assert len(overlap) == 0, "Patient leakage detected!"

In [ ]:














# =========================================================
# 8. DICOM -> PIL IMAGE
# =========================================================
def dicom_to_uint8(path):
    dcm = pydicom.dcmread(path)
    arr = dcm.pixel_array.astype(np.float32)

    # Apply VOI LUT if available would be nice, but keeping it simple here
    arr = arr - arr.min()
    if arr.max() > 0:
        arr = arr / arr.max()

    arr = (arr * 255).clip(0, 255).astype(np.uint8)
    return arr

# =========================================================
# 9. DATASET
# =========================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class MammogramViewDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = dicom_to_uint8(row["dicom_path"])

        # Convert grayscale to 3-channel PIL image
        img = Image.fromarray(img).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = int(row["label"])
        patient_id = str(row[patient_col])

        return img, label, patient_id

train_dataset = MammogramViewDataset(train_df, transform=train_transform)
val_dataset = MammogramViewDataset(val_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# =========================================================
# 10. MODEL: ViT-B/16
# =========================================================
num_classes = len(classes)

weights = models.ViT_B_16_Weights.IMAGENET1K_V1
model = models.vit_b_16(weights=weights)

in_features = model.heads.head.in_features
model.heads.head = nn.Linear(in_features, num_classes)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# =========================================================
# 11. TRAIN / VALIDATE
# =========================================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels, _ in tqdm(loader, desc="Training", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels, _ in tqdm(loader, desc="Validating", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc, np.array(all_labels), np.array(all_preds)

# =========================================================
# 12. RUN TRAINING
# =========================================================
best_val_acc = -1
best_model_path = f"vit_b16_{VIEW_TO_USE.lower()}_assessment_best.pth"

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, y_true, y_pred = evaluate(model, val_loader, criterion, DEVICE)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved best model to {best_model_path}")

# =========================================================
# 13. FINAL REPORT
# =========================================================
print(f"\nBest validation accuracy: {best_val_acc:.4f}")

# Reload best model for clean final eval
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
_, final_val_acc, y_true, y_pred = evaluate(model, val_loader, criterion, DEVICE)

print(f"\nFinal Val Accuracy: {final_val_acc:.4f}")

target_names = [f"BI-RADS {idx_to_class[i]}" for i in range(num_classes)]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))